In [ ]:
!pip install pyspark
import builtins
import shutil
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, median, to_timestamp, hour, month, when
from pyspark.sql.functions import avg, max as spark_max, min as spark_min, count, to_date, round as spark_round
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

# Increase driver memory to handle collecting the large enriched dataset to pandas
spark = (
    SparkSession.builder
    .appName("Weather-Data")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)


In [ ]:
# ─────────────────────────────────────────────
# EXTRACT – Load raw hourly weather data from CSV
# ─────────────────────────────────────────────

# Define schema explicitly to ensure correct data types on load
schema = StructType([
    StructField("city_name",                        StringType(), True),
    StructField("datetime",                         StringType(), True),
    StructField("temperature_2m",                   DoubleType(), True),
    StructField("relative_humidity_2m",             DoubleType(), True),
    StructField("dew_point_2m",                     DoubleType(), True),
    StructField("apparent_temperature",             DoubleType(), True),
    StructField("precipitation",                    DoubleType(), True),
    StructField("rain",                             DoubleType(), True),
    StructField("snowfall",                         DoubleType(), True),
    StructField("snow_depth",                       DoubleType(), True),
    StructField("weather_code",                     DoubleType(), True),
    StructField("pressure_msl",                     DoubleType(), True),
    StructField("surface_pressure",                 DoubleType(), True),
    StructField("cloud_cover",                      DoubleType(), True),
    StructField("cloud_cover_low",                  DoubleType(), True),
    StructField("cloud_cover_mid",                  DoubleType(), True),
    StructField("cloud_cover_high",                 DoubleType(), True),
    StructField("et0_fao_evapotranspiration",       DoubleType(), True),
    StructField("vapour_pressure_deficit",          DoubleType(), True),
    StructField("wind_speed_10m",                   DoubleType(), True),
    StructField("wind_speed_100m",                  DoubleType(), True),
    StructField("wind_direction_10m",               DoubleType(), True),
    StructField("wind_direction_100m",              DoubleType(), True),
    StructField("wind_gusts_10m",                   DoubleType(), True),
    StructField("soil_temperature_0_to_7cm",        DoubleType(), True),
    StructField("soil_temperature_7_to_28cm",       DoubleType(), True),
    StructField("soil_temperature_28_to_100cm",     DoubleType(), True),
    StructField("soil_temperature_100_to_255cm",    DoubleType(), True),
    StructField("soil_moisture_0_to_7cm",           DoubleType(), True),
    StructField("soil_moisture_7_to_28cm",          DoubleType(), True),
    StructField("soil_moisture_28_to_100cm",        DoubleType(), True),
    StructField("soil_moisture_100_to_255cm",       DoubleType(), True),
    StructField("shortwave_radiation",              DoubleType(), True),
    StructField("direct_radiation",                 DoubleType(), True),
    StructField("diffuse_radiation",                DoubleType(), True),
    StructField("direct_normal_irradiance",         DoubleType(), True),
    StructField("global_tilted_irradiance",         DoubleType(), True),
    StructField("terrestrial_radiation",            DoubleType(), True),
    StructField("shortwave_radiation_instant",      DoubleType(), True),
    StructField("direct_radiation_instant",         DoubleType(), True),
    StructField("diffuse_radiation_instant",        DoubleType(), True),
    StructField("direct_normal_irradiance_instant", DoubleType(), True),
    StructField("global_tilted_irradiance_instant", DoubleType(), True),
    StructField("terrestrial_radiation_instant",    DoubleType(), True),
])

# Load CSV using the defined schema
raw_df = spark.read.csv("hourly_data.csv", header=True, schema=schema)

print("CSV loaded! Total rows:", raw_df.count())
print("Columns:", len(raw_df.columns))


In [ ]:
# ─────────────────────────────────────────────
# EDA BEFORE PREPROCESSING
# ─────────────────────────────────────────────
import matplotlib.pyplot as plt

print("1. Descriptive Statistics (Raw Data):")
# Showing stats for a few key columns to check for outliers or issues
raw_df.select("temperature_2m", "relative_humidity_2m", "precipitation").describe().show()

print("2. Univariate Graphical Analysis (Raw Data):")
# Convert a specific column to Pandas for Matplotlib visualization [cite: 66, 286, 288]
# Note: Dropping NA just for the plot so matplotlib doesn't throw an error on raw nulls
raw_temp_pd = raw_df.select("temperature_2m").dropna().toPandas()

plt.figure(figsize=(8, 5))
plt.hist(raw_temp_pd['temperature_2m'], bins=30, color='red', alpha=0.7, edgecolor='black')
plt.title("Distribution of Temperature (Before Preprocessing)", fontdict={'family': 'serif', 'size': 15})
plt.xlabel("Temperature (°C)")
plt.ylabel("Frequency")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

1. Descriptive Statistics (Raw Data):


NameError: name 'raw_df' is not defined

In [ ]:
# ─────────────────────────────────────────────
# TRANSFORM – Step 1: Inspect the data
# ─────────────────────────────────────────────

print("─" * 100)
print("HOURLY WEATHER DATASET")
print("─" * 100)

# Display the structure of the DataFrame
raw_df.printSchema()

# Display first 10 rows of raw data before preprocessing
print("First 10 rows of raw data:")
raw_df.show(10, truncate=False)

# Count nulls per column BEFORE cleaning using count(when(...)) pattern
print("\nNull count per column (before cleaning):")
raw_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in raw_df.columns
]).show(truncate=False)


In [ ]:
# ─────────────────────────────────────────────
# TRANSFORM – Step 2: Fill missing values
# ─────────────────────────────────────────────

# All numeric measurement columns – nulls will be replaced with their column mean
numeric_cols = [
    "temperature_2m", "relative_humidity_2m", "dew_point_2m", "apparent_temperature",
    "precipitation", "rain", "snowfall", "snow_depth", "weather_code",
    "pressure_msl", "surface_pressure", "cloud_cover", "cloud_cover_low",
    "cloud_cover_mid", "cloud_cover_high", "et0_fao_evapotranspiration",
    "vapour_pressure_deficit", "wind_speed_10m", "wind_speed_100m",
    "wind_direction_10m", "wind_direction_100m", "wind_gusts_10m",
    "soil_temperature_0_to_7cm", "soil_temperature_7_to_28cm",
    "soil_temperature_28_to_100cm", "soil_temperature_100_to_255cm",
    "soil_moisture_0_to_7cm", "soil_moisture_7_to_28cm",
    "soil_moisture_28_to_100cm", "soil_moisture_100_to_255cm",
    "shortwave_radiation", "direct_radiation", "diffuse_radiation",
    "direct_normal_irradiance", "global_tilted_irradiance", "terrestrial_radiation",
    "shortwave_radiation_instant", "direct_radiation_instant", "diffuse_radiation_instant",
    "direct_normal_irradiance_instant", "global_tilted_irradiance_instant",
    "terrestrial_radiation_instant",
]

print("─" * 100)
print("DATA PREPROCESSING")
print("─" * 100)
print("1. FILLING MISSING / NULL VALUES")

# Compute the mean of each numeric column; use builtins.round to avoid
# conflict with PySpark's round function
fill_numeric = {}
for c in numeric_cols:
    avg_val = raw_df.select(avg(col(c))).collect()[0][0]
    if avg_val is not None:
        fill_numeric[c] = builtins.round(float(avg_val), 2)

print("Fill values (mean per column):", fill_numeric)

# Replace null numeric values with their column mean
df_filled = raw_df.fillna(fill_numeric)

# Replace null string values with "Unknown"
df_filled = df_filled.fillna({"city_name": "Unknown"})

print("Missing values filled!")


In [ ]:
# ─────────────────────────────────────────────
# TRANSFORM – Step 3: Remove duplicates and invalid records
# ─────────────────────────────────────────────

# Drop rows where city_name is null (key identifier must be present)
# Then remove exact duplicate rows with distinct()
df_clean = df_filled.dropna(subset=["city_name"]).distinct()

print("\n2. REMOVING DUPLICATES AND INVALID VALUES")
print("Rows before deduplication:", df_filled.count())
print("Rows after  deduplication:", df_clean.count())

# Verify nulls are gone after cleaning
print("\nNull count per column (after cleaning):")
df_clean.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_clean.columns
]).show(truncate=False)

print("─" * 100)
print("CLEANED DATAFRAME RESULT")
print("─" * 100)
print("Number of rows in cleaned DataFrame:", df_clean.count())
print("Cleaned DataFrame (first 10 rows):")
df_clean.show(10, truncate=False)


In [ ]:
# ─────────────────────────────────────────────
# TRANSFORM – Step 4: Filter unwanted records
# Remove rows that fall outside physically plausible ranges for
# a tropical/subtropical dataset (Philippines context)
# ─────────────────────────────────────────────

filtered_df = df_clean.filter(
    # Temperature in °C: plausible range 10 – 50
    (col("temperature_2m") >= 10) & (col("temperature_2m") <= 50) &
    # Relative humidity must be 0–100 %
    (col("relative_humidity_2m") >= 0) & (col("relative_humidity_2m") <= 100) &
    # Precipitation cannot be negative
    (col("precipitation") >= 0) &
    # Wind speed cannot be negative
    (col("wind_speed_10m") >= 0) &
    # Sea-level pressure: typical range 900–1100 hPa
    (col("pressure_msl") >= 900) & (col("pressure_msl") <= 1100) &
    # city_name must not be empty
    (col("city_name") != "") &
    # datetime is a string here – check for null before type conversion
    col("datetime").isNotNull()
)

print(f"Rows after filtering invalid records : {filtered_df.count():,}")


In [ ]:
# ─────────────────────────────────────────────
# TRANSFORM – Step 5: Convert data types
# - datetime string  → TimestampType
# - weather_code     → IntegerType  (stored as float by inferSchema)
# - snow_depth       → DoubleType   (ensure consistent numeric type)
# ─────────────────────────────────────────────

typed_df = (
    filtered_df
    # Parse datetime string to a proper Timestamp
    .withColumn("datetime", to_timestamp(col("datetime"), "yyyy-MM-dd HH:mm:ss"))
    # weather_code is a numeric categorical – cast to Integer
    .withColumn("weather_code", col("weather_code").cast(IntegerType()))
    # Ensure snow_depth is stored as Double
    .withColumn("snow_depth", col("snow_depth").cast(DoubleType()))
)

print("Updated schema after type conversions:")
typed_df.select("datetime", "weather_code", "snow_depth").printSchema()
typed_df.select("datetime", "weather_code", "snow_depth").show(3)


In [ ]:
# ─────────────────────────────────────────────
# TRANSFORM – Step 6: Create new columns
# Derived features that add analytical value:
#   temperature_f      – Temperature converted to Fahrenheit
#   is_raining         – Boolean flag: rain > 0
#   heat_category      – Categorical label based on apparent temperature
#   hour               – Hour of the day extracted from datetime
#   month              – Month extracted from datetime
# ─────────────────────────────────────────────


enriched_df = (
    typed_df
    # Convert Celsius to Fahrenheit
    .withColumn("temperature_f", (col("temperature_2m") * 9 / 5) + 32)

    # Boolean flag for active rainfall
    .withColumn("is_raining", col("rain") > 0)

    # Heat category based on apparent (feels-like) temperature in °C
    .withColumn(
        "heat_category",
        when(col("apparent_temperature") < 26, "Cool")
        .when((col("apparent_temperature") >= 26) & (col("apparent_temperature") < 32), "Comfortable")
        .when((col("apparent_temperature") >= 32) & (col("apparent_temperature") < 38), "Hot")
        .otherwise("Very Hot")
    )

    # Extract time components for time-series grouping
    .withColumn("hour", hour(col("datetime")))
    .withColumn("month", month(col("datetime")))
)

enriched_df.select(
    "city_name", "datetime", "temperature_2m", "temperature_f",
    "is_raining", "heat_category", "hour", "month"
).show(5)


In [ ]:
# ─────────────────────────────────────────────
# EDA AFTER PREPROCESSING
# ─────────────────────────────────────────────
import matplotlib.pyplot as plt

print("1. Descriptive Statistics (Cleaned Data):")
enriched_df.select("temperature_2m", "relative_humidity_2m", "precipitation").describe().show()

print("2. Univariate Graphical Analysis (Cleaned Data):")
# Convert cleaned column to Pandas
clean_temp_pd = enriched_df.select("temperature_2m").toPandas()

plt.figure(figsize=(8, 5))
plt.hist(clean_temp_pd['temperature_2m'], bins=30, color='blue', alpha=0.7, edgecolor='black')
plt.title("Distribution of Temperature (After Preprocessing)", fontdict={'family': 'serif', 'size': 15})
plt.xlabel("Temperature (°C)")
plt.ylabel("Frequency")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()


print("3. Bivariate Graphical Analysis (Cleaned Data):")
# Let's compare Temperature vs Apparent Temperature (Feels Like) [cite: 32, 263, 748, 749]
# We'll take a random sample so the scatter plot isn't overly crowded
sample_df = enriched_df.sample(fraction=0.01, seed=42).select("temperature_2m", "apparent_temperature").toPandas()

plt.figure(figsize=(8, 5))
# Using scatter plot to see relationship between two variables
plt.scatter(sample_df['temperature_2m'], sample_df['apparent_temperature'], marker='o', color='green', alpha=0.5)
plt.title("Temperature vs Apparent Temperature", fontdict={'family': 'serif', 'size': 15})
plt.xlabel("Actual Temperature (°C)")
plt.ylabel("Apparent Temperature (°C)")
plt.grid(True, linestyle='--')
plt.show()

In [ ]:
# ─────────────────────────────────────────────
# TRANSFORM – Step 7: Aggregate data
# Compute daily summary statistics per city:
#   avg_temp_c         – Daily average temperature (°C)
#   max_temp_c         – Daily maximum temperature (°C)
#   min_temp_c         – Daily minimum temperature (°C)
#   total_precipitation– Total daily precipitation (mm)
#   avg_humidity       – Daily average relative humidity (%)
#   max_wind_speed     – Daily maximum wind speed at 10 m (km/h)
#   rainy_hours        – Count of hours with active rainfall
# ─────────────────────────────────────────────


daily_df = (
    enriched_df
    # Derive a date column for daily grouping
    .withColumn("date", to_date(col("datetime")))
    .groupBy("city_name", "date")
    .agg(
        spark_round(avg("temperature_2m"), 2).alias("avg_temp_c"),
        spark_round(spark_max("temperature_2m"), 2).alias("max_temp_c"),
        spark_round(spark_min("temperature_2m"), 2).alias("min_temp_c"),
        spark_round(spark_sum("precipitation"), 2).alias("total_precipitation_mm"),
        spark_round(avg("relative_humidity_2m"), 2).alias("avg_humidity_pct"),
        spark_round(spark_max("wind_speed_10m"), 2).alias("max_wind_speed_kmh"),
        spark_sum(col("is_raining").cast("int")).alias("rainy_hours")
    )
    .orderBy("city_name", "date")
)

print(f"Daily aggregate rows : {daily_df.count():,}")
daily_df.show(10)


In [ ]:
# ─────────────────────────────────────────────
# EXPORT – Step 8: Save first 500,000 cleaned rows to Parquet
# Use a stable sort so the first 500k rows are deterministic.
# ─────────────────────────────────────────────

cleaned_500k_df = (
    enriched_df
    .orderBy("city_name", "datetime")
    .limit(500_000)
)

parquet_dir = Path("cleaned_first_500k_tmp")
parquet_file = Path("cleaned_first_500k.parquet")

if parquet_dir.exists():
    shutil.rmtree(parquet_dir)

if parquet_file.exists():
    parquet_file.unlink()

(
    cleaned_500k_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .parquet(str(parquet_dir))
)

part_file = next(parquet_dir.glob("part-*.parquet"))
shutil.copy2(part_file, parquet_file)
shutil.rmtree(parquet_dir)

print(f"Cleaned 500k rows saved : {cleaned_500k_df.count():,}")
print(f"Parquet file created     : {parquet_file.resolve()}")
